# DSC 411/511
# k-Nearest Neighbours on Ames Housing


---
# Section 0: Setup and the Target

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

sns.set_style('whitegrid')
SEED = 42
print('ready')

ready


In [10]:
DATA_path= "/Users/home/workspace/School/DataMining/DataMining/class_workbooks/datasets/AmesHousing.csv"

df = pd.read_csv(DATA_path)

print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

Rows: 2930
Columns: 82


In [11]:
# Same target as last class: above or below the median sale price
median_price = df['SalePrice'].median()
df['Expensive'] = (df['SalePrice'] > median_price).astype(int)

DROP = ['Order', 'PID', 'SalePrice', 'Expensive']
num = df.select_dtypes(include='number').drop(columns=[c for c in DROP if c in df.columns])

X = num.fillna(num.median())
y = df['Expensive']

print(f'Median SalePrice : ${median_price:,.0f}')
print(f'Features         : {X.shape[1]} numeric columns')
print(f'Rows             : {X.shape[0]:,}')
print()
print('Class balance:')
print((y.value_counts(normalize=True) * 100).round(1)
        .rename({0: 'Below median', 1: 'Above median'}).to_string())
print()
print('SalePrice in X?', 'SalePrice' in X.columns, ' <- must be False, or this is leakage')

Median SalePrice : $160,000
Features         : 36 numeric columns
Rows             : 2,930

Class balance:
Expensive
Below median    50.1
Above median    49.9

SalePrice in X? False  <- must be False, or this is leakage


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)

baseline = max(y_test.mean(), 1 - y_test.mean())
print(f'Training rows : {len(X_train):,}')
print(f'Test rows     : {len(X_test):,}')
print(f'Baseline      : {baseline*100:.1f}%   (always guess the majority class)')

Training rows : 2,197
Test rows     : 733
Baseline      : 50.1%   (always guess the majority class)


---
# Section 1: k-NN With No Scaling

k-NN compares columns against each other. Look at what that means here.

In [13]:
rng = X.max() - X.min()
print('Column ranges, widest first:')
print(rng.sort_values(ascending=False).head(6).round(0).to_string())
print()
print('Narrowest:')
print(rng.sort_values().head(5).round(0).to_string())

Column ranges, widest first:
Lot Area         213945.0
Misc Val          17000.0
Total Bsmt SF      6110.0
BsmtFin SF 1       5644.0
Gr Liv Area        5308.0
1st Flr SF         4761.0

Narrowest:
Bsmt Half Bath    2.0
Half Bath         2.0
Kitchen AbvGr     3.0
Bsmt Full Bath    3.0
Fireplaces        4.0


In [14]:
# Take two real houses and see what the distance is actually made of
a, b = X.iloc[0], X.iloc[1]
diff_sq = (a - b) ** 2
total = diff_sq.sum()

share = (diff_sq / total * 100).sort_values(ascending=False)
print('Distance between house 0 and house 1, broken down by column:')
print(f"{'column':22}{'difference':>14}{'squared':>16}{'share':>10}")
for c in share.head(6).index:
    print(f'{c:22}{(a[c]-b[c]):>14,.1f}{diff_sq[c]:>16,.0f}{share[c]:>9.2f}%')
print()
print(f'Total squared distance: {total:,.0f}')
print(f'Euclidean distance    : {np.sqrt(total):,.1f}')
print()
print('One column is almost the entire distance. That column is not "most important".')
print('It simply happens to be measured in the biggest numbers.')

Distance between house 0 and house 1, broken down by column:
column                    difference         squared     share
Lot Area                    20,148.0     405,941,904    99.67%
1st Flr SF                     760.0         577,600     0.14%
Gr Liv Area                    760.0         577,600     0.14%
Garage Area                   -202.0          40,804     0.01%
Total Bsmt SF                  198.0          39,204     0.01%
BsmtFin SF 1                   171.0          29,241     0.01%

Total squared distance: 407,295,756
Euclidean distance    : 20,181.6

One column is almost the entire distance. That column is not "most important".
It simply happens to be measured in the biggest numbers.


In [15]:
knn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
acc_raw = accuracy_score(y_test, knn_raw.predict(X_test))

print(f'k-NN (k=5) on RAW data : {acc_raw*100:.1f}%')
print(f'Baseline               : {baseline*100:.1f}%')
print()
print('Remember this number. Section 2 runs the identical model on scaled data.')

k-NN (k=5) on RAW data : 83.8%
Baseline               : 50.1%

Remember this number. Section 2 runs the identical model on scaled data.


---
# Section 2: The Same Model, Scaled

Subtract the mean, divide by the standard deviation. Every column now contributes
on the same terms.

**Important:** fit the scaler on the training data only. Fitting it on everything
lets information from the test set leak into training.

In [16]:
scaler = StandardScaler().fit(X_train)       # fit on TRAIN only
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
acc_scaled = accuracy_score(y_test, knn_scaled.predict(X_test_s))

print(f'k-NN (k=5), RAW    : {acc_raw*100:.1f}%')
print(f'k-NN (k=5), SCALED : {acc_scaled*100:.1f}%')
print(f'Change             : {(acc_scaled-acc_raw)*100:+.1f} points')
print()
print('Same algorithm. Same k. Same rows. Only the units changed.')

k-NN (k=5), RAW    : 83.8%
k-NN (k=5), SCALED : 90.7%
Change             : +7.0 points

Same algorithm. Same k. Same rows. Only the units changed.


In [17]:
# The same breakdown after scaling
a_s = pd.Series(scaler.transform(X.iloc[[0]])[0], index=X.columns)
b_s = pd.Series(scaler.transform(X.iloc[[1]])[0], index=X.columns)
dsq = (a_s - b_s) ** 2
tot = dsq.sum()
sh = (dsq / tot * 100).sort_values(ascending=False)

print('Same two houses, now scaled. Share of the distance:')
for c in sh.head(6).index:
    print(f'   {c:22}{sh[c]:>8.2f}%')
print()
print('No single column dominates any more. Every column gets a fair hearing.')

Same two houses, now scaled. Share of the distance:
   Fireplaces               20.24%
   Lot Frontage             17.08%
   Lot Area                 12.81%
   Screen Porch              9.06%
   Bsmt Full Bath            7.71%
   1st Flr SF                7.66%

No single column dominates any more. Every column gets a fair hearing.


In [18]:
# Who are house 0's nearest neighbours, before and after scaling?
def neighbours(Xtr, query, k=5):
    m = KNeighborsClassifier(n_neighbors=k).fit(Xtr, y_train)
    dist, idx = m.kneighbors(query, n_neighbors=k)
    return idx[0], dist[0]

q_raw = X_test.iloc[[0]]
q_s   = X_test_s[[0]]

i_raw, d_raw = neighbours(X_train, q_raw)
i_s,   d_s   = neighbours(X_train_s, q_s)

print('Nearest neighbours of the first test house')
print()
print('RAW    ->', list(X_train.index[i_raw]))
print('SCALED ->', list(X_train.index[i_s]))
print()
overlap = set(i_raw) & set(i_s)
print(f'Houses appearing in BOTH lists: {len(overlap)} of 5')
print()
print('Scaling did not adjust the answer slightly. It chose different neighbours.')

print('RAW:')
for idx, dist in zip(X_train.index[i_raw], d_raw):
    print(f'   house {idx:>5}   distance {dist:>12,.1f}')

print('SCALED:')
for idx, dist in zip(X_train.index[i_s], d_s):
    print(f'   house {idx:>5}   distance {dist:>8.2f}')

Nearest neighbours of the first test house

RAW    -> [1224, 2654, 1908, 2837, 2639]
SCALED -> [2697, 2197, 2235, 1224, 1416]

Houses appearing in BOTH lists: 1 of 5

Scaling did not adjust the answer slightly. It chose different neighbours.
RAW:
   house  1224   distance        351.9
   house  2654   distance        383.3
   house  1908   distance        385.6
   house  2837   distance        459.3
   house  2639   distance        479.4
SCALED:
   house  2697   distance     2.57
   house  2197   distance     2.68
   house  2235   distance     2.77
   house  1224   distance     2.81
   house  1416   distance     2.83


---
# Section 3: Choosing k

In [19]:
ks = [1, 2, 3, 5, 7, 9, 11, 15, 21, 31, 51, 101, 201, 401, 801]
rows = []
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)
    rows.append({
        'k': k,
        'train acc': round(accuracy_score(y_train, m.predict(X_train_s)) * 100, 1),
        'test acc' : round(accuracy_score(y_test,  m.predict(X_test_s))  * 100, 1),
    })
kt = pd.DataFrame(rows)
kt['gap'] = (kt['train acc'] - kt['test acc']).round(1)
print(kt.to_string(index=False))

  k  train acc  test acc  gap
  1      100.0      88.4 11.6
  2       94.2      87.9  6.3
  3       94.5      89.8  4.7
  5       93.4      90.7  2.7
  7       92.9      91.3  1.6
  9       92.8      90.9  1.9
 11       92.3      91.1  1.2
 15       92.0      91.4  0.6
 21       91.5      90.7  0.8
 31       90.9      91.0 -0.1
 51       90.5      90.2  0.3
101       89.6      89.8 -0.2
201       89.2      89.1  0.1
401       89.2      89.1  0.1
801       89.1      89.1  0.0


In [20]:
# Cross validation is the honest way to pick k. One split can mislead you.
cv_rows = []
for k in [1, 3, 5, 7, 9, 11, 15, 21, 31, 51, 101, 201]:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    s = cross_val_score(pipe, X_train, y_train, cv=5)
    cv_rows.append({'k': k, 'CV mean': round(s.mean()*100, 1),
                    'CV std': round(s.std()*100, 2)})
cv = pd.DataFrame(cv_rows)
print(cv.to_string(index=False))
print()
best_k = int(cv.loc[cv['CV mean'].idxmax(), 'k'])
print(f'Best k by cross validation: {best_k}')
print()
print('Note the pipeline. StandardScaler is INSIDE it, so the scaler is refitted')
print('on each training fold. Scaling outside the pipeline leaks the fold into itself.')

  k  CV mean  CV std
  1     89.1    1.11
  3     89.9    1.38
  5     90.7    1.54
  7     90.9    1.30
  9     91.0    1.51
 11     90.8    2.03
 15     91.0    1.77
 21     90.6    1.72
 31     90.2    2.01
 51     89.9    1.96
101     89.1    1.22
201     88.9    1.09

Best k by cross validation: 9

Note the pipeline. StandardScaler is INSIDE it, so the scaler is refitted
on each training fold. Scaling outside the pipeline leaks the fold into itself.


---
# Section 4: Euclidean or Manhattan?

Both are Minkowski distance with a different setting of r.

In [ ]:
rows = []
for name, p in [('Manhattan (r=1)', 1), ('Euclidean (r=2)', 2), ('Minkowski r=3', 3)]:
    for k in [5, best_k, 21]:
        m = KNeighborsClassifier(n_neighbors=k, p=p).fit(X_train_s, y_train)
        rows.append({'metric': name, 'k': k,
                     'test acc': round(accuracy_score(y_test, m.predict(X_test_s))*100, 1)})
mt = pd.DataFrame(rows).pivot(index='metric', columns='k', values='test acc')
print('Test accuracy by distance measure and k')
print(mt.to_string())
print()
print('The differences are usually small. The measure matters far less than scaling.')

---
# Section 5: Do We Run PCA First?

Arguments both ways. We do not guess. We run all four combinations.

In [ ]:
pca_full = PCA().fit(X_train_s)
cum = np.cumsum(pca_full.explained_variance_ratio_)

for t in [0.80, 0.90, 0.95, 0.99]:
    k_ = int(np.argmax(cum >= t) + 1)
    print(f'  {int(t*100)}% of variance : {k_:>2} of {X.shape[1]} components')

In [ ]:
setups = {}

setups['Raw, no scaling'] = KNeighborsClassifier(n_neighbors=best_k).fit(X_train, y_train)
setups['Scaled']          = KNeighborsClassifier(n_neighbors=best_k).fit(X_train_s, y_train)

results = [
    {'input': 'Raw, no scaling', 'columns': X.shape[1],
     'test acc': round(accuracy_score(y_test, setups['Raw, no scaling'].predict(X_test))*100, 1)},
    {'input': 'Scaled', 'columns': X.shape[1],
     'test acc': round(accuracy_score(y_test, setups['Scaled'].predict(X_test_s))*100, 1)},
]

for t in [0.80, 0.90, 0.95]:
    ncomp = int(np.argmax(cum >= t) + 1)
    p = PCA(n_components=ncomp).fit(X_train_s)
    Ztr, Zte = p.transform(X_train_s), p.transform(X_test_s)
    m = KNeighborsClassifier(n_neighbors=best_k).fit(Ztr, y_train)
    results.append({'input': f'Scaled + PCA ({int(t*100)}% var)', 'columns': ncomp,
                    'test acc': round(accuracy_score(y_test, m.predict(Zte))*100, 1)})

p2 = PCA(n_components=2).fit(X_train_s)
m2 = KNeighborsClassifier(n_neighbors=best_k).fit(p2.transform(X_train_s), y_train)
results.append({'input': 'Scaled + PCA (2 comp)', 'columns': 2,
                'test acc': round(accuracy_score(y_test, m2.predict(p2.transform(X_test_s)))*100, 1)})

res = pd.DataFrame(results)
res['vs scaled'] = (res['test acc'] - res.loc[1, 'test acc']).round(1)
print(f'k = {best_k} throughout. Baseline {baseline*100:.1f}%')
print()
print(res.to_string(index=False))

In [ ]:
plt.figure(figsize=(9.5, 4.4))
cols = ['#C00000'] + ['#595959'] + ['#0070C0'] * (len(res) - 2)
plt.bar(res['input'], res['test acc'], color=cols, edgecolor='black')
plt.axhline(res.loc[1, 'test acc'], color='black', ls='--', lw=1, label='scaled, all columns')
plt.axhline(baseline*100, color='grey', ls=':', lw=1, label='baseline')
plt.ylabel('Test accuracy (%)'); plt.ylim(40, 100)
plt.xticks(rotation=20, ha='right')
plt.title(f'What should we feed k-NN?   (k = {best_k})')
plt.legend(); plt.tight_layout(); plt.show()

### Read your own numbers before concluding anything

Two things are pulling against each other:

- **PCA removes correlated, redundant columns.** Two columns measuring the same
  thing would otherwise count twice in the distance.
- **PCA also throws information away.** Whatever the discarded components held
  is gone.

Which effect wins depends on the dataset.

**What is not in doubt:** scaling. Look at the gap between the first two bars.

---
# Section 6: No Training, Slow Prediction

Every other model learns something and then answers quickly. k-NN does the opposite.

In [ ]:
m = KNeighborsClassifier(n_neighbors=best_k)

t0 = time.perf_counter(); m.fit(X_train_s, y_train);  t_fit  = time.perf_counter() - t0
t0 = time.perf_counter(); _ = m.predict(X_test_s);    t_pred = time.perf_counter() - t0

print(f'Training {len(X_train):,} rows : {t_fit*1000:>8.1f} ms')
print(f'Predicting {len(X_test):,} rows: {t_pred*1000:>8.1f} ms')
print()
print(f'Prediction took {t_pred/max(t_fit,1e-9):.0f}x longer than "training".')
print()
print('Training just stored the data. Nothing was learned.')
print(f'Each prediction compares one house against all {len(X_train):,} stored houses.')

In [ ]:
# How does prediction time grow with the size of the training set?
rows = []
for frac in [0.1, 0.25, 0.5, 1.0]:
    n = int(len(X_train) * frac)
    mm = KNeighborsClassifier(n_neighbors=best_k).fit(X_train_s[:n], y_train.iloc[:n])
    t0 = time.perf_counter(); mm.predict(X_test_s); dt = time.perf_counter() - t0
    rows.append({'training rows': n, 'predict time (ms)': round(dt*1000, 1)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print('This is why k-NN does not scale. Ames has 2,930 rows and it is fine.')
print('At ten million rows, every single prediction is a scan of ten million rows.')

---
# Section 7: Three Methods, One Question

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    'Decision tree (depth 5)': (DecisionTreeClassifier(max_depth=5, random_state=SEED), False),
    'Naive Bayes (Gaussian)' : (GaussianNB(), False),
    f'k-NN (k={best_k}), unscaled': (KNeighborsClassifier(n_neighbors=best_k), False),
    f'k-NN (k={best_k}), scaled'  : (KNeighborsClassifier(n_neighbors=best_k), True),
}

rows = []
for name, (mdl, needs_scale) in models.items():
    Xtr, Xte = (X_train_s, X_test_s) if needs_scale else (X_train, X_test)
    mdl.fit(Xtr, y_train)
    pipe = make_pipeline(StandardScaler(), mdl) if needs_scale else mdl
    cv_in = X_train if needs_scale else X_train
    rows.append({'model': name,
                 'test acc': round(accuracy_score(y_test, mdl.predict(Xte))*100, 1),
                 '5-fold CV': round(cross_val_score(pipe, cv_in, y_train, cv=5).mean()*100, 1)})

comp = pd.DataFrame(rows)
print(f'Baseline: {baseline*100:.1f}%')
print()
print(comp.to_string(index=False))

In [ ]:
plt.figure(figsize=(9.5, 4.3))
cols = ['#0070C0', '#2E7D32', '#C00000', '#FF8C00']
plt.bar(comp['model'], comp['test acc'], color=cols, edgecolor='black')
plt.axhline(baseline*100, color='black', ls='--', lw=1, label='baseline')
plt.ylabel('Test accuracy (%)'); plt.ylim(40, 100)
plt.xticks(rotation=18, ha='right')
plt.title('Same question, same data, three different methods')
plt.legend(); plt.tight_layout(); plt.show()